In [34]:
import pdfplumber   # reads PDF files and extracts text from them
import os           # helps load API key
import json         # converts JSON strings 

In [35]:
pdf_path = "./Invoices/invoice-3.pdf"   # path to your PDF file

table_count = 0    # counter to track how many tables we find
table_data = []    # list to store all extracted tables

with pdfplumber.open(pdf_path) as pdf:          # open the PDF safely using 'with'
    for i, page in enumerate(pdf.pages):        # loop through every page
        if page.extract_table():                # check if this page has a table
            table_data.append(page.extract_table())  # store the table
            table_count += 1                    # increment the counter

print(f"Total number of tables found: {table_count}")

Total number of tables found: 2


In [36]:
table_1 = table_data[0]   
print("Raw table output:")
print(table_1)

print("\nRow by row:")
for row in table_1:
    print(row)

Raw table output:
[['Product', 'Unit Price [EUR]', 'Total [EUR]'], ['Glossostigma\nQty. 3', '9.90', '29.70'], ['Bayberry\nQty. 222', '5.44', '1,207.68'], ['Waxflower\nQty. 34', '1.67', '56.78'], ['Carolina Geranium\nQty. 45', '4.17', '187.65'], ["Smooth Solomon's\nSeal\nQty. 3", '4.97', '14.91'], ['El Yunque Colorado\nQty. 23', '9.97', '229.31']]

Row by row:
['Product', 'Unit Price [EUR]', 'Total [EUR]']
['Glossostigma\nQty. 3', '9.90', '29.70']
['Bayberry\nQty. 222', '5.44', '1,207.68']
['Waxflower\nQty. 34', '1.67', '56.78']
['Carolina Geranium\nQty. 45', '4.17', '187.65']
["Smooth Solomon's\nSeal\nQty. 3", '4.97', '14.91']
['El Yunque Colorado\nQty. 23', '9.97', '229.31']


In [37]:
pdf_path = "./Invoices/invoice-3.pdf"

data = ""    # we start with an empty string and keep adding to it

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        if page.extract_text():                       # only process if text exists
            data += page.extract_text() + "\n"       # add each page's text + newline

# Preview the first 100 characters
print(data[:100])

Invoice No. 1213
16.12.2021
To
La Galerie
4 Rue Courtois
Lille, Nord, 59000
Ship To
Same as recipien


In [38]:
def clean_text(text):
    """
    Cleans raw PDF text by removing empty lines and extra whitespace.
    
    Input:  raw string (as returned by pdfplumber)
    Output: clean string ready to send to the LLM
    """
    
    # Split into individual lines
    lines = text.split("\n")
    
    # Strip whitespace and remove empty lines
    cleaned_lines = [line.strip() for line in lines if line.strip()]
    
    # Join back into one clean string
    cleaned = "\n".join(cleaned_lines)
    
    return cleaned

In [39]:
cleaned_content = clean_text(data)

print(f"Before cleaning: {len(data)} characters")
print(f"After cleaning:  {len(cleaned_content)} characters")
print(f"\nPreview:\n{cleaned_content[:300]}")

Before cleaning: 780 characters
After cleaning:  779 characters

Preview:
Invoice No. 1213
16.12.2021
To
La Galerie
4 Rue Courtois
Lille, Nord, 59000
Ship To
Same as recipient
Instructions
None
Product Unit Price [EUR] Total [EUR]
Glossostigma
Qty. 3 9.90 29.70
Bayberry
Qty. 222 5.44 1,207.68
Waxflower
Qty. 34 1.67 56.78
Carolina Geranium
Qty. 45 4.17 187.65
Smooth Solomo


In [40]:
system_prompt = """You are a precise document data extractor.

Your task is to extract specific information from the document text provided by the user.

Return your response as a valid JSON object with EXACTLY these fields:
{
  "document_type": "the type of document (invoice, contract, resume, report, etc.)",
  "title": "the main title or heading of the document",
  "date": "any date mentioned in the document (format: DD-MM-YYYY if possible)",
  "author_or_sender": "the name of the person or company who created or sent this document",
  "recipient": "the name of the person or company this document is addressed to",
  "key_points": ["list", "of", "3 to 5", "most important points from the document"],
  "total_amount": "any monetary total or amount mentioned (null if not applicable)",
  "summary": "a 2-3 sentence plain English summary of what this document is about"
}

Rules:
- Return ONLY the JSON object. No explanation. No markdown. No code blocks.
- If a field cannot be found in the document, set its value to null.
- Do not guess or make up values. Only extract what is clearly present.
"""

print("✅ System prompt ready.")

✅ System prompt ready.


In [41]:
!pip install openai
!pip install dotenv
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

response = client.chat.completions.create(
    model="gemma-4-31b-it:free",
    messages=[
        {
            "role": "system",
            "content": system_prompt          # our extraction instructions
        },
        {
            "role": "user",
            "content": f"Here is the document text to extract from:\n\n{cleaned_content}"
                                              # the actual PDF text
        }
    ],
    temperature=0                             # deterministic — same output every time
)

# Extract the text content from the response
raw_response = response.choices[0].message.content

print("✅ Response received")
print(f"Type: {type(raw_response)}")
print(f"\nRaw response:\n{raw_response}")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ Response received
Type: <class 'str'>

Raw response:
{
  "document_type": "invoice",
  "title": "Invoice No. 1213",
  "date": "16-12-2021",
  "author_or_sender": "Zencorporations",
  "recipient": "La Galerie",
  "key_points": [
    "Invoice number 1213 for various plant products",
    "Total amount due is 2809.30 EUR",
    "Payment is due 30 days after receipt",
    "Includes sales tax of 255.39 EUR"
  ],
  "total_amount": "2809.30",
  "summary": "This is a sales invoice from Zencorporations to La Galerie for the purchase of various plants. The total amount due, including sales tax, is 2809.30 EUR."
}


In [42]:
import json

# json.loads() = JSON load from string
# Converts a JSON-formatted string into a Python dictionary
dict_data = json.loads(raw_response)

print(f"Type after parsing: {type(dict_data)}")
print(f"\nParsed dictionary:\n{dict_data}")

Type after parsing: <class 'dict'>

Parsed dictionary:
{'document_type': 'invoice', 'title': 'Invoice No. 1213', 'date': '16-12-2021', 'author_or_sender': 'Zencorporations', 'recipient': 'La Galerie', 'key_points': ['Invoice number 1213 for various plant products', 'Total amount due is 2809.30 EUR', 'Payment is due 30 days after receipt', 'Includes sales tax of 255.39 EUR'], 'total_amount': '2809.30', 'summary': 'This is a sales invoice from Zencorporations to La Galerie for the purchase of various plants. The total amount due, including sales tax, is 2809.30 EUR.'}


In [43]:
with open("invoice_data_3.json", "w") as file:
    json.dump(dict_data, file, indent=4)

print("JSON file created successfully!")

JSON file created successfully!
